# MIRA-Bench annotation

Downloads the official `ziqiangoodgood/MIRAGE` dataset repository and annotates the 100 MIRA-Bench samples. Its JSONL contains each composite instruction and five per-edit polygon masks. For each mask, the LLM parses the full instruction with that region highlighted and selects the corresponding source reference; masks ground the regions but do not replace LLM parsing. Each parsed reference becomes a visual token (`<v1>`–`<v5>`). Images have no edited target image, so `target_ref` remains null. The MIRA-Bench paper and source repository are available at [arXiv](https://arxiv.org/abs/2604.05180) and [GitHub](https://github.com/ZiqianLiu666/MIRAGE).

Install from the repository root with `uv sync`, copy `.env.example` to `.env`, and set `RANDOM_SEED` plus `GEMINI_API_KEY` (or `GOOGLE_API_KEY`). New runs use `.out/annotation/mira_bench/run_<timestamp>/`. Existing canonical runs are resumed automatically; if only legacy `.out/annotation/mirage/` runs exist, the latest is resumed in place using its existing filenames and stable sample IDs.

In [1]:
import json
import os
from dataclasses import replace
from functools import partial
from pathlib import Path

from huggingface_hub import snapshot_download

from annotation import (
    AnnotationConfig,
    convert_rows,
    create_annotation_run_directory,
    mira_bench,
    resolve_annotation_run_directory,
)

DATASET_ID = "ziqiangoodgood/MIRAGE"
SPLIT = "test"
CONFIG = AnnotationConfig.from_env("mira_bench")
LEGACY_OUTPUT_DIR = CONFIG.output_dir.parent / "mirage"
LEGACY_RUNS = sorted(
    path
    for path in LEGACY_OUTPUT_DIR.glob("run_*")
    if path.is_dir() and (path / "mirage_mixed.jsonl").is_file()
)
USE_LEGACY_OUTPUT = not os.environ.get("ANNOTATE_OUT") and not any(
    path.is_dir() for path in CONFIG.output_dir.glob("run_*")
) and bool(LEGACY_RUNS)
if USE_LEGACY_OUTPUT:
    CONFIG = replace(CONFIG, output_dir=LEGACY_OUTPUT_DIR)
OUTPUT_NAME = "mirage_mixed.jsonl" if USE_LEGACY_OUTPUT else "mira_bench_mixed.jsonl"
AUDIT_NAME = "mirage_audit.jsonl" if USE_LEGACY_OUTPUT else "mira_bench_audit.jsonl"

DATASET_SNAPSHOT = Path(
    snapshot_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        allow_patterns=[
            "benchmark/annotations.jsonl",
            "benchmark/*.jpg",
            "benchmark/*.jpeg",
            "benchmark/*.png",
            "benchmark/*.webp",
        ],
    )
)
BENCHMARK_ROOT = DATASET_SNAPSHOT / "benchmark"
ANNOTATION_FILE = BENCHMARK_ROOT / "annotations.jsonl"
with ANNOTATION_FILE.open(encoding="utf-8") as stream:
    ROWS = [json.loads(line) for line in stream if line.strip()]

if not ROWS:
    raise ValueError(f"No MIRA-Bench annotations found in {ANNOTATION_FILE}")
print(f"Loaded {len(ROWS)} MIRA-Bench rows from {ANNOTATION_FILE}")

/Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 601 files: 100%|██████████| 601/601 [00:00<00:00, 7737.97it/s]

Loaded 100 MIRA-Bench rows from /Users/nghialt3670/.cache/huggingface/hub/datasets--ziqiangoodgood--MIRAGE/snapshots/11eff1e3f396e189e61bd1f0ca596286d8a0b183/benchmark/annotations.jsonl


The released MIRA-Bench JSONL stores `image`, a composite `editing_instruction`, and five ordered target-region polygons in `mask`; it does not include `bbox` or `refer_object` fields. The adapter builds a separate mask per edit. The instruction is split into its five edit clauses and one LLM call sees the image with all five masks numbered and colored, matching every mask to exactly one clause (a one-to-one assignment, since mask order does not always follow clause order). An invalid matching is retried once and then audited. Instructions that do not split into five clauses fall back to parsing the full instruction once per highlighted mask, with a MIRA-Bench-specific retry. Each region mask is retained for its reference crop and mask asset. Attempts are capped by `ANNOTATE_LIMIT` from `.env` (defaults to 10 per run; set it to `0` to process all remaining rows). Set `SAMPLE_INDICES` (zero-based row positions, e.g. `[4, 10, 30, 57]`) in the run cell to annotate only those rows; with `OVERWRITE_SELECTED = True` their existing records are replaced.

In [2]:
SAMPLE_INDICES = None  # e.g. [4, 10, 30, 57] to annotate only those rows.
SAMPLE_INDICES = [4, 10, 30, 57]
OVERWRITE_SELECTED = True  # Replace existing records of the selected rows.
if SAMPLE_INDICES is not None:
    invalid = [i for i in SAMPLE_INDICES if not 0 <= i < len(ROWS)]
    if invalid:
        raise ValueError(f"SAMPLE_INDICES out of range: {invalid}")
OVERWRITE_IDS = (
    {f"mirage:{ROWS[i]['image']}" for i in SAMPLE_INDICES}
    if SAMPLE_INDICES is not None and OVERWRITE_SELECTED
    else None
)

RESUME_LAST_RUN = False
RUN_DIR = (
    resolve_annotation_run_directory(CONFIG.output_dir)
    if RESUME_LAST_RUN
    else create_annotation_run_directory(CONFIG.output_dir)
)
RUN_CONFIG = replace(CONFIG, output_dir=RUN_DIR)
print("Annotation run directory:", RUN_DIR)

summary = convert_rows(
    rows=ROWS,
    adapter=partial(mira_bench, image_root=BENCHMARK_ROOT),
    split=SPLIT,
    annotator=RUN_CONFIG.create_annotator(use_grounder=False),
    output=RUN_DIR / OUTPUT_NAME,
    failures=RUN_DIR / AUDIT_NAME,
    limit=None if SAMPLE_INDICES is not None else RUN_CONFIG.limit,
    overwrite_sample_ids=OVERWRITE_IDS,
    row_indices=SAMPLE_INDICES,
)
summary

Annotation run directory: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/annotation/mira_bench/run_20261017_153758_826510


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
test: 100it [00:28,  3.47it/s]


{'accepted': 4, 'skipped': 0, 'failed': 0, 'resumed': 0, 'retried': 0}